In [3]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

# CONFIGURATION
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB = 20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
STEMS = ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0

## Building Dataset

In [48]:
def build_dataset(root_dir, val_split=0.17, seed=42):

    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    
    rng = random.Random(seed)
    
    corrupted_count = 0  
    size_less_than_5MB = 0
    size_greater_than_5MB = 0
    
    genres_path = os.path.join(root_dir, 'genres_stems')
    
    for genre in GENRES:
        genre_path = os.path.join(genres_path, genre)
   
        valid_songs = []
        
        for song_folder in [f for f in os.listdir(genre_path)]:
            song_path = os.path.join(genre_path, song_folder)
                    
            is_corrupted = False
            for stem in STEMS:
                stem_path = os.path.join(song_path, stem)
                file_size_bytes = os.path.getsize(stem_path)
                file_size_kb = file_size_bytes / 1024
                file_size_mb = file_size_bytes / (1024 * 1024)
                
                # for Q1 and Q2
                if file_size_kb < 4:
                    is_corrupted = True
                    corrupted_count += 1
                
                if file_size_mb < 5.0491:
                    size_less_than_5MB += 1
                if file_size_mb > 5.0493:
                    size_greater_than_5MB += 1
            
            if not is_corrupted:
                valid_songs.append(song_folder)
        
        # Stratified Shuffle Split
        rng.shuffle(valid_songs)
        val_size = int(len(valid_songs) * val_split)
        val_songs = valid_songs[:val_size]
        train_songs = valid_songs[val_size:]
        
        def add_to_dict(target_dict, song_list):
            for song_folder in song_list:
                song_path = os.path.join(genre_path, song_folder)
                for stem in STEMS:
                    stem_name = stem.replace('.wav', '')
                    stem_path = os.path.join(song_path, stem)
                    target_dict[genre][stem_name].append(stem_path)
        
        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    print(f"Corrupted files (< 4kb): {corrupted_count}")
    print(f"Files < (5.0491MB): {size_less_than_5MB}")
    print(f"Files > (5.0493MB): {size_greater_than_5MB}")
    print(f"\nQ1 Ans: {corrupted_count} + {size_less_than_5MB} = {corrupted_count + size_less_than_5MB}")
    print(f"Q2 Ans: |{size_greater_than_5MB} - {size_less_than_5MB}| = {abs(size_greater_than_5MB - size_less_than_5MB)}")
    
    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

Corrupted files (< 4kb): 0
Files < (5.0491MB): 1256
Files > (5.0493MB): 184

Q1 Ans: 0 + 1256 = 1256
Q2 Ans: |184 - 1256| = 1072


### Q1) What is the value of total number of corrupted sounds ( less than 4kb) + (total number of sounds < 5.0491MB) ? 
### Ans: **1256**

### Q2) What is the absolute difference between  total number of sounds > 5.0493MB and total number of sounds < 5.0491MB ? 
### Ans: **1072**

In [20]:
train_reggae_drums = len(tr['reggae']['drums'])
val_country_vocals = len(val['country']['vocals'])
    
diff = abs(train_reggae_drums - val_country_vocals)

print(f"Training reggae drums samples: {train_reggae_drums}")
print(f"Validation country vocals samples: {val_country_vocals}")
print(f"Q3 Ans: |{train_reggae_drums}-{val_country_vocals}| = {diff}")

Training reggae drums samples: 83
Validation country vocals samples: 17
Q3 Ans: |83-17| = 66


### Q3) What is the absolute difference between the number of training reggae drum samples and the number of validation country vocal samples? 
### Ans: **66**

## Finding long silences in training data

In [22]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):

    records = []
    total_files = 0
    for genre in dataset_dict:
        for stem_name in dataset_dict[genre]:
            total_files += len(dataset_dict[genre][stem_name])
    
    print(f"Total files: {total_files}")
    
    file_count = 0
    for genre in tqdm(dataset_dict, desc="Processing genres"):
        for stem_name in dataset_dict[genre]:
            for file_path in dataset_dict[genre][stem_name]:
                file_count += 1
                
                try:
                    y, _ = librosa.load(file_path, sr=sr)
                    total_duration = len(y) / sr

                    non_silent_intervals = librosa.effects.split(y, top_db=top_db)
                    
                    max_silence = 0.0
                    silence_type = []
                    
                    # CASE A: Fully silent
                    if len(non_silent_intervals) == 0:
                        max_silence = total_duration
                        silence_type.append("full")
                    else:
                        # CASE B: START silence
                        start_silence = non_silent_intervals[0][0] / sr
                        if start_silence >= threshold_sec:
                            silence_type.append("start")
                        if start_silence > max_silence:
                            max_silence = start_silence
                        
                        # CASE C: END silence
                        end_silence = (len(y) - non_silent_intervals[-1][1]) / sr
                        if end_silence >= threshold_sec:
                            silence_type.append("end")
                        if end_silence > max_silence:
                            max_silence = end_silence
                        
                        # CASE D: MIDDLE silence
                        for i in range(len(non_silent_intervals) - 1):
                            gap_start = non_silent_intervals[i][1]
                            gap_end = non_silent_intervals[i + 1][0]
                            gap_duration = (gap_end - gap_start) / sr
                            
                            if gap_duration >= threshold_sec:
                                if "middle" not in silence_type:
                                    silence_type.append("middle")
                            if gap_duration > max_silence:
                                max_silence = gap_duration
                    
                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 2),
                            "Silence_Location": ", ".join(silence_type) if silence_type else "none",
                            "File_Path": file_path
                        })
                
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")
    
    df = pd.DataFrame(records)
    return df

df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

Total files: 3320


Processing genres: 100%|██████████| 10/10 [06:09<00:00, 36.99s/it]


### Q4) Total number of sound files having silence greater than equal to 5 secs : **658**

In [24]:
print(f"Total files with silence >= 5 secs: {len(df_silence)}")

Total files with silence >= 5 secs: 658


### Q5) Total number of sound tracks in Vocals where silence >= 5 secs : **304**


In [29]:
vocals_silence = df_silence[df_silence['Stem'] == 'vocals']
print(f"Total Vocals tracks with silence >= 5 secs: {len(vocals_silence)}")

Total Vocals tracks with silence >= 5 secs: 304


### Q6) What's the average Silence Length in Vocals (in secs) ? **12.79 secs**

In [30]:
vsl = vocals_silence['Max_Silence_Sec'].mean()
print(f"Average silence length in Vocals: {vsl:.2f} secs")

Average silence length in Vocals: 12.79 secs


### Q7) Total number of drums sound tracks in jazz where silence >= 5 secs : **19**


In [31]:
jazz_drums = df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')]
print(f"Jazz drums with silence >= 5 secs: {len(jazz_drums)}")

Jazz drums with silence >= 5 secs: 19


### Q8) Total number of drums sound tracks in jazz where silence >= 5 secs and Silence_Location is only middle : **11**

In [32]:
jd_middle = jazz_drums[jazz_drums['Silence_Location'] == 'middle']
print(f"Jazz drums with silence >= 5 secs and Silence_Location is ONLY middle: {len(jd_middle)}")

Jazz drums with silence >= 5 secs and Silence_Location is ONLY middle: 11


### Q9) Total number of drums sound tracks in jazz where silence >= 5 secs and Max_Silence_Sec >= 10 : **5**

In [35]:
jd_10sec = jazz_drums[jazz_drums['Max_Silence_Sec'] >= 10]
print(f"Jazz drums with silence >=5secs and Max_Silence_Sec >=10 : {len(jd_10sec)}")

Jazz drums with silence >=5secs and Max_Silence_Sec >=10 : 5


In [39]:
print("Pivot Table: Genre vs Stem\n")
pivot_table = pd.pivot_table(df_silence, values='File_Path', index='Genre',columns='Stem', aggfunc='count', fill_value=0)
print(pivot_table)

Pivot Table: Genre vs Stem

Stem       bass  drums  other  vocals
Genre                                
blues        16     23      5      41
classical    69     57      3      70
country      14     14      2      14
disco         6      2      2      17
hiphop       21      3     21       5
jazz         21     19      1      73
metal         4      0      1      37
pop          10      6      2       4
reggae        4      5      7      13
rock          9      7      0      30


## Stems mixing and Analyis

In [ ]:
stems_audio = []
    
try:
    for key in STEM_KEYS:
        file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
        
        y, _ = librosa.load(file_path, sr=SR, duration=DURATION)
        stems_audio.append(y)
        
    print("Audio loaded successfully.")

except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")

except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")

except Exception as e:
    print(f"ERROR: {e}")
        
    
# Ensure all stems have the same length
min_length = min(len(stem) for stem in stems_audio)
stems_audio = [stem[:min_length] for stem in stems_audio]
        
# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.stack(stems_audio, axis=0)
print(f"\nStems stack shape: {stems_stack.shape}")
        
# Mix the stems
mix_raw = np.sum(stems_stack, axis=0)
print(f"Q10: Length of mix sample: {len(mix_raw)}")
        
# RMS Amplitude
rms_val = np.sqrt(np.mean(mix_raw ** 2))
print(f"Q11: RMS Amplitude of mix sample: {rms_val:.2f}")
        
# Peak Normalization
max_val = np.max(np.abs(mix_raw))
print(f"Max absolute value before normalization: {max_val:.2f}")
        
if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw
        
# Max value of peak normalized sample
print(f"Q12: Max value of peak normalized sample: {round(np.max(np.abs(mix_norm)), 2)}")
        
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."

Audio loaded successfully.

Stems stack shape: (4, 110250)
Q10: Length of mix sample: 110250
Q11: RMS Amplitude of mix sample: 0.11
Max absolute value before normalization: 0.53
Q12: Max value of peak normalized sample: 1.0


### Q10) What is the length of the mix sample? 
### Ans: **110250**

### Q11) What is the value of RMS Amplitude of mix sample? 
### Ans: **0.11**

### Q12) What is the value of max value of peak  normalized sample ? 
### Ans: **1.0**